# OpenEvolve Circle Packing Tutorial on Colab

This notebook is a self-contained Colab version of the OpenEvolve circle-packing tutorial.

It installs OpenEvolve, starts a local open-source LLM through an OpenAI-compatible endpoint, defines the circle-packing evaluator and score functions, runs OpenEvolve, and visualizes checkpoint updates.

Recommended Colab setting: `Runtime -> Change runtime type -> T4 GPU` or better.

Colab GPU availability varies over time. If the local LLM cannot start, the score-function verification and fallback visualization cells still work.

References:

- Google Colab FAQ: https://research.google.com/colaboratory/faq.html
- llama-cpp-python OpenAI-compatible server: https://llama-cpp-python.readthedocs.io/en/latest/server/
- Qwen2.5-Coder-7B-Instruct GGUF: https://huggingface.co/bartowski/Qwen2.5-Coder-7B-Instruct-GGUF

## 0. Parameters

For the first run, keep the defaults: `PACKING_N=16`, `SCORE_MODE=hard_valid_sum`, `ITERATIONS=2`.

After the first successful run, change one parameter at a time and rerun from section 6 onward.

In [ ]:
#@title Tutorial parameters
PACKING_N = "16"  #@param ["10", "16", "26"]
PACKING_N = int(PACKING_N)

SCORE_MODE = "hard_valid_sum"  #@param ["hard_valid_sum", "soft_penalty", "intentionally_bad_reported_sum"]
ITERATIONS = 2  #@param {type:"integer"}
TEMPERATURE = 0.4  #@param {type:"slider", min:0.1, max:1.2, step:0.1}
MAX_TOKENS = "1536"  #@param ["1024", "1536", "2048"]
MAX_TOKENS = int(MAX_TOKENS)

MODEL_REPO = "bartowski/Qwen2.5-Coder-7B-Instruct-GGUF"
MODEL_FILE = "Qwen2.5-Coder-7B-Instruct-Q4_K_M.gguf"
MODEL_ALIAS = "local-qwen2.5-coder-7b"
LLM_API_BASE = "http://127.0.0.1:8000/v1"
LLM_PORT = 8000
N_CTX = 8192
N_GPU_LAYERS = -1

WORKDIR = "/content/openevolve_colab_tutorial"
RUN_ROOT = "/content/openevolve_runs"

print({
    "PACKING_N": PACKING_N,
    "SCORE_MODE": SCORE_MODE,
    "ITERATIONS": ITERATIONS,
    "TEMPERATURE": TEMPERATURE,
    "MAX_TOKENS": MAX_TOKENS,
    "MODEL": f"{MODEL_REPO}/{MODEL_FILE}",
})

## 1. Runtime Check

In [ ]:
import platform
import subprocess
import sys

print("Python:", sys.version)
print("Platform:", platform.platform())
try:
    result = subprocess.run(["nvidia-smi"], text=True, capture_output=True, check=False)
    print(result.stdout if result.returncode == 0 else result.stderr)
except FileNotFoundError:
    print("nvidia-smi not found. You may be on CPU; local LLM inference will be slow.")

## 2. Install Packages

This can take several minutes, especially the `llama-cpp-python` install.

In [ ]:
%%bash
set -euo pipefail

echo "[1/4] Upgrading pip tooling..."
python -m pip install --upgrade pip setuptools wheel

echo "[2/4] Installing base Python packages..."
python -m pip install numpy scipy matplotlib pandas pyyaml openai huggingface_hub requests ipython

echo "[3/4] Base package installation completed"

if python - <<'PY' >/dev/null 2>&1
import llama_cpp
PY
then
  echo "[4/4] llama-cpp-python already installed"
else
  if command -v nvidia-smi >/dev/null 2>&1; then
    echo "[4/4] Installing llama-cpp-python with CUDA build flags. This can take several minutes."
    CMAKE_ARGS="-DGGML_CUDA=on" FORCE_CMAKE=1 python -m pip install --upgrade --force-reinstall --no-cache-dir "llama-cpp-python[server]" || \
      (echo "CUDA build failed; falling back to CPU llama-cpp-python" && CMAKE_ARGS="-DGGML_CUDA=off" FORCE_CMAKE=1 python -m pip install --upgrade --force-reinstall --no-cache-dir "llama-cpp-python[server]")
  else
    echo "[4/4] Installing CPU llama-cpp-python. Live OpenEvolve will be slow."
    CMAKE_ARGS="-DGGML_CUDA=off" FORCE_CMAKE=1 python -m pip install --upgrade --force-reinstall --no-cache-dir "llama-cpp-python[server]"
  fi
fi


## 3. Clone and Install OpenEvolve

In [ ]:
%%bash
set -euo pipefail
if [ ! -d /content/openevolve/.git ]; then
  git clone --depth 1 https://github.com/algorithmicsuperintelligence/openevolve.git /content/openevolve
fi
python -m pip install -q -e /content/openevolve
python - <<'PY'
import openevolve
print("OpenEvolve import OK:", getattr(openevolve, "__version__", "unknown"))
PY

## 4. Start a Local OpenAI-Compatible LLM

This downloads a Qwen2.5-Coder 7B Q4 GGUF model and starts `llama-cpp-python` as a local server.

If this fails, skip to section 7. You can still inspect the score functions and fallback plots.

In [ ]:
import subprocess
import sys
import time
from pathlib import Path

import requests
from huggingface_hub import hf_hub_download

model_dir = Path("/content/models")
model_dir.mkdir(parents=True, exist_ok=True)
print("Downloading model. First download is about 4-5 GB.")
model_path = hf_hub_download(
    repo_id=MODEL_REPO,
    filename=MODEL_FILE,
    local_dir=str(model_dir),
    local_dir_use_symlinks=False,
)
print("Model path:", model_path)

try:
    LLM_SERVER_PROC.terminate()
    LLM_SERVER_PROC.wait(timeout=10)
except NameError:
    pass
except Exception:
    try:
        LLM_SERVER_PROC.kill()
    except Exception:
        pass

server_log_path = Path("/content/llama_cpp_server.log")
server_log = open(server_log_path, "w")
cmd = [
    sys.executable, "-m", "llama_cpp.server",
    "--model", str(model_path),
    "--host", "127.0.0.1",
    "--port", str(LLM_PORT),
    "--n_ctx", str(N_CTX),
    "--n_gpu_layers", str(N_GPU_LAYERS),
    "--chat_format", "chatml",
]
print("Starting:", " ".join(cmd))
LLM_SERVER_PROC = subprocess.Popen(cmd, stdout=server_log, stderr=subprocess.STDOUT, text=True)

models_url = f"http://127.0.0.1:{LLM_PORT}/v1/models"
for second in range(240):
    if LLM_SERVER_PROC.poll() is not None:
        server_log.flush()
        print(server_log_path.read_text(errors="replace")[-4000:])
        raise RuntimeError("LLM server exited early")
    try:
        response = requests.get(models_url, timeout=2)
        if response.ok:
            print("LLM server ready:", response.json())
            break
    except Exception:
        pass
    if second % 10 == 0:
        print(f"waiting... {second}s")
    time.sleep(1)
else:
    server_log.flush()
    print(server_log_path.read_text(errors="replace")[-4000:])
    raise TimeoutError("LLM server did not become ready")

## 5. LLM Health Check

In [ ]:
from openai import OpenAI
client = OpenAI(base_url=LLM_API_BASE, api_key="not-needed")
response = client.chat.completions.create(
    model=MODEL_ALIAS,
    messages=[{"role": "user", "content": "Return only the word ok."}],
    max_tokens=16,
    temperature=0,
)
print(response.choices[0].message.content)

## 6. Create the Circle-Packing Problem

The evaluator has a single `evaluate()` path and `cascade_evaluation: false`, so the active score mode is exactly the score used by OpenEvolve.

In [ ]:
from pathlib import Path

problem_dir = Path(WORKDIR) / "circle_packing"
problem_dir.mkdir(parents=True, exist_ok=True)
Path(RUN_ROOT).mkdir(parents=True, exist_ok=True)

embedded_files = {
  "initial_program.py": "# EVOLVE-BLOCK-START\nimport math\nimport os\nimport numpy as np\n\n\ndef construct_packing():\n    n = int(os.environ.get(\"PACKING_N\", \"16\"))\n    cols = int(math.ceil(math.sqrt(n)))\n    rows = int(math.ceil(n / cols))\n    dx = 1.0 / cols\n    dy = 1.0 / rows\n    radius = 0.35 * min(dx, dy)\n    centers = []\n    for k in range(n):\n        row = k // cols\n        col = k % cols\n        centers.append(((col + 0.5) * dx, (row + 0.5) * dy))\n    centers = np.asarray(centers, dtype=float)\n    radii = np.full(n, radius, dtype=float)\n    return centers, radii, float(np.sum(radii))\n# EVOLVE-BLOCK-END\n\n\ndef run_packing():\n    return construct_packing()\n",
  "hacked_program.py": "# EVOLVE-BLOCK-START\nimport os\nimport numpy as np\n\n\ndef construct_packing():\n    n = int(os.environ.get(\"PACKING_N\", \"16\"))\n    centers = np.full((n, 2), 0.5, dtype=float)\n    radii = np.full(n, 0.5, dtype=float)\n    reported_sum = 999.0\n    return centers, radii, reported_sum\n# EVOLVE-BLOCK-END\n\n\ndef run_packing():\n    return construct_packing()\n",
  "evaluator.py": "\"\"\"\nTutorial evaluator for circle packing (n=26), RAW-score version.\n\nDifferences from the upstream example evaluator\n------------------------------------------------\n1. `combined_score` is the RAW objective (sum of radii), NOT normalized by the\n   AlphaEvolve reference value 2.635. This makes the tutorial message honest:\n   \"we maximize the sum of radii; 2.635 is a reference, not a proven optimum.\"\n\n2. There is a SINGLE scoring path: `evaluate()`. The cascade stage functions\n   (`evaluate_stage1` / `evaluate_stage2`) are intentionally REMOVED.\n\n   Why this matters (verified against OpenEvolve 0.3.0):\n   - When `cascade_evaluation: true`, the engine calls `evaluate_stage1/2` and\n     BYPASSES `evaluate()` (openevolve/evaluator.py:163-165, 388-389).\n   - Thresholds are compared directly against `combined_score`, un-normalized\n     (openevolve/evaluator.py:689).\n   So a raw-score change placed only in `evaluate()` would silently have no\n   effect under cascade. By removing the stage functions, even if someone flips\n   `cascade_evaluation: true`, the engine falls back to `_direct_evaluate` ->\n   `evaluate()` (openevolve/evaluator.py:388-389). Use `cascade_evaluation: false`\n   in config_tutorial.yaml regardless.\n\nScore modes (select via the SCORE_MODE environment variable)\n------------------------------------------------------------\n- hard_valid_sum (default):\n      combined_score = actual_sum        if the packing is geometrically valid\n                     = 0.0               otherwise\n  The honest objective. Invalid geometry scores 0 no matter how large the sum.\n\n- soft_penalty:\n      combined_score = reported_sum\n                       - OVERLAP_WEIGHT  * overlap_penalty\n                       - BOUNDARY_WEIGHT * boundary_penalty\n  Trusts the program's *reported* sum but subtracts continuous penalties for\n  overlaps and out-of-square parts. Shows that a badly-shaped reward is gameable\n  and can even go negative.\n\n- intentionally_bad_reported_sum:\n      combined_score = reported_sum      (NO validation at all)\n  The \"reward hacking\" demo: a program can simply return an inflated third value\n  and win. Illustrates exactly why you need an independent, honest evaluator.\n\nThe number of circles is read from PACKING_N (default 26) so the same file can be\nreused for the Intermediate tier; the geometric checks are already n-agnostic.\n\"\"\"\n\nimport numpy as np\nimport os\nimport pickle\nimport subprocess\nimport sys\nimport tempfile\nimport time\nimport traceback\n\n\nclass TimeoutError(Exception):\n    pass\n\n\ndef _config():\n    \"\"\"Read tutorial knobs from the environment (dashboard sets these per job).\"\"\"\n    n = int(os.environ.get(\"PACKING_N\", \"26\"))\n    score_mode = os.environ.get(\"SCORE_MODE\", \"hard_valid_sum\")\n    overlap_weight = float(os.environ.get(\"OVERLAP_WEIGHT\", \"1.0\"))\n    boundary_weight = float(os.environ.get(\"BOUNDARY_WEIGHT\", \"1.0\"))\n    return n, score_mode, overlap_weight, boundary_weight\n\n\n# AlphaEvolve reported value for n=26. Reference ONLY (not a proven optimum,\n# and only meaningful when n == 26). Never used for selection.\nREFERENCE_VALUE_N26 = 2.635\n\n\ndef validate_packing(centers, radii):\n    \"\"\"\n    Validate that circles don't overlap and are inside the unit square.\n\n    Returns True if valid, False otherwise.\n    \"\"\"\n    n = centers.shape[0]\n\n    if np.isnan(centers).any() or np.isnan(radii).any():\n        print(\"NaN values detected in centers/radii\")\n        return False\n\n    for i in range(n):\n        if radii[i] < 0:\n            print(f\"Circle {i} has negative radius {radii[i]}\")\n            return False\n\n    for i in range(n):\n        x, y = centers[i]\n        r = radii[i]\n        if x - r < -1e-6 or x + r > 1 + 1e-6 or y - r < -1e-6 or y + r > 1 + 1e-6:\n            print(f\"Circle {i} at ({x}, {y}) r={r} is outside the unit square\")\n            return False\n\n    for i in range(n):\n        for j in range(i + 1, n):\n            dist = np.sqrt(np.sum((centers[i] - centers[j]) ** 2))\n            if dist < radii[i] + radii[j] - 1e-6:\n                print(f\"Circles {i} and {j} overlap: dist={dist}, r1+r2={radii[i] + radii[j]}\")\n                return False\n\n    return True\n\n\ndef compute_penalties(centers, radii):\n    \"\"\"\n    Continuous (non-binary) constraint violations, used by the soft_penalty mode.\n\n    overlap_penalty : total linear overlap depth summed over all circle pairs.\n    boundary_penalty: total distance by which circles poke outside the unit square\n                      (a negative radius or an out-of-[0,1] center inflates this).\n    \"\"\"\n    n = centers.shape[0]\n    overlap_penalty = 0.0\n    boundary_penalty = 0.0\n\n    # NaN-safe: treat NaN as a large violation so the reward never rewards it.\n    if np.isnan(centers).any() or np.isnan(radii).any():\n        return 1e6, 1e6\n\n    for i in range(n):\n        x, y = centers[i]\n        r = radii[i]\n        # Distance from center to the nearest wall; negative if the center is\n        # already outside the square. A circle is inside iff r <= wall_dist.\n        wall_dist = min(x, y, 1 - x, 1 - y)\n        boundary_penalty += max(0.0, r - wall_dist)\n        # A negative radius is itself a violation.\n        boundary_penalty += max(0.0, -r)\n\n    for i in range(n):\n        for j in range(i + 1, n):\n            dist = np.sqrt(np.sum((centers[i] - centers[j]) ** 2))\n            overlap_penalty += max(0.0, (radii[i] + radii[j]) - dist)\n\n    return float(overlap_penalty), float(boundary_penalty)\n\n\ndef run_with_timeout(program_path, timeout_seconds=30):\n    \"\"\"\n    Run `run_packing()` from program_path in a fresh subprocess with a timeout.\n\n    Returns (centers, radii, reported_sum). The third value is whatever the\n    program CLAIMS its sum is (may be a lie -- that's the point of some modes).\n    \"\"\"\n    with tempfile.NamedTemporaryFile(suffix=\".py\", delete=False) as temp_file:\n        script = f\"\"\"\nimport sys, os, pickle, traceback\nimport numpy as np\nsys.path.insert(0, os.path.dirname({program_path!r}))\ntry:\n    import importlib.util\n    spec = importlib.util.spec_from_file_location(\"program\", {program_path!r})\n    program = importlib.util.module_from_spec(spec)\n    spec.loader.exec_module(program)\n    centers, radii, sum_radii = program.run_packing()\n    with open({temp_file.name!r} + \".results\", \"wb\") as f:\n        pickle.dump({{\"centers\": centers, \"radii\": radii, \"sum_radii\": sum_radii}}, f)\nexcept Exception as e:\n    traceback.print_exc()\n    with open({temp_file.name!r} + \".results\", \"wb\") as f:\n        pickle.dump({{\"error\": str(e)}}, f)\n\"\"\"\n        temp_file.write(script.encode())\n        temp_file_path = temp_file.name\n\n    results_path = f\"{temp_file_path}.results\"\n    try:\n        process = subprocess.Popen(\n            [sys.executable, temp_file_path],\n            stdout=subprocess.PIPE,\n            stderr=subprocess.PIPE,\n        )\n        try:\n            stdout, stderr = process.communicate(timeout=timeout_seconds)\n            if process.returncode != 0:\n                if stderr:\n                    print(f\"Subprocess stderr: {stderr.decode()}\")\n                raise RuntimeError(f\"Process exited with code {process.returncode}\")\n            if not os.path.exists(results_path):\n                raise RuntimeError(\"Results file not found\")\n            with open(results_path, \"rb\") as f:\n                results = pickle.load(f)\n            if \"error\" in results:\n                raise RuntimeError(f\"Program execution failed: {results['error']}\")\n            return results[\"centers\"], results[\"radii\"], results[\"sum_radii\"]\n        except subprocess.TimeoutExpired:\n            process.kill()\n            process.wait()\n            raise TimeoutError(f\"Process timed out after {timeout_seconds} seconds\")\n    finally:\n        for p in (temp_file_path, results_path):\n            if os.path.exists(p):\n                os.unlink(p)\n\n\ndef _zero_metrics(eval_time, error=None):\n    # Numeric-only metrics (plus an optional 'error' string on failure, matching\n    # the upstream example). The active score_mode is logged, not returned, so a\n    # full evolution run never carries a non-numeric feature.\n    m = {\n        \"combined_score\": 0.0,\n        \"sum_radii\": 0.0,\n        \"reported_sum\": 0.0,\n        \"validity\": 0.0,\n        \"overlap_penalty\": 0.0,\n        \"boundary_penalty\": 0.0,\n        \"eval_time\": float(eval_time),\n    }\n    if error is not None:\n        m[\"error\"] = str(error)\n    return m\n\n\ndef evaluate(program_path):\n    \"\"\"Single scoring path (no cascade stages). Returns a metrics dict.\"\"\"\n    n, score_mode, overlap_weight, boundary_weight = _config()\n    start_time = time.time()\n\n    try:\n        centers, radii, reported_sum = run_with_timeout(program_path, timeout_seconds=30)\n    except Exception as e:\n        print(f\"Evaluation failed to run program: {e}\")\n        return _zero_metrics(time.time() - start_time, error=e)\n\n    centers = np.asarray(centers, dtype=float)\n    radii = np.asarray(radii, dtype=float)\n\n    # Shape check. A wrong shape means we cannot score geometry at all.\n    if centers.shape != (n, 2) or radii.shape != (n,):\n        print(f\"Invalid shapes: centers={centers.shape}, radii={radii.shape}, expected n={n}\")\n        m = _zero_metrics(time.time() - start_time, error=\"invalid shapes\")\n        # intentionally_bad still rewards the reported value even here.\n        if score_mode == \"intentionally_bad_reported_sum\":\n            m[\"reported_sum\"] = float(reported_sum)\n            m[\"combined_score\"] = float(reported_sum)\n        return m\n\n    valid = validate_packing(centers, radii)\n    actual_sum = float(np.sum(radii))\n    overlap_penalty, boundary_penalty = compute_penalties(centers, radii)\n    eval_time = time.time() - start_time\n\n    if score_mode == \"hard_valid_sum\":\n        combined_score = actual_sum if valid else 0.0\n\n    elif score_mode == \"soft_penalty\":\n        combined_score = (\n            float(reported_sum)\n            - overlap_weight * overlap_penalty\n            - boundary_weight * boundary_penalty\n        )\n\n    elif score_mode == \"intentionally_bad_reported_sum\":\n        # No validation on purpose: trust whatever the program reports.\n        combined_score = float(reported_sum)\n\n    else:\n        raise ValueError(f\"Unknown SCORE_MODE: {score_mode!r}\")\n\n    metrics = {\n        \"combined_score\": float(combined_score),\n        \"sum_radii\": float(actual_sum),        # honest, recomputed from radii\n        \"reported_sum\": float(reported_sum),   # what the program claimed\n        \"validity\": 1.0 if valid else 0.0,\n        \"overlap_penalty\": float(overlap_penalty),\n        \"boundary_penalty\": float(boundary_penalty),\n        \"eval_time\": float(eval_time),\n    }\n    # Reference ratio is display-only and meaningful only for n == 26.\n    if n == 26:\n        metrics[\"reference_ratio\"] = float(actual_sum / REFERENCE_VALUE_N26)\n\n    print(\n        f\"Evaluation: mode={score_mode}, valid={valid}, actual_sum={actual_sum:.6f}, \"\n        f\"reported_sum={float(reported_sum):.6f}, combined_score={combined_score:.6f}, \"\n        f\"overlap={overlap_penalty:.4f}, boundary={boundary_penalty:.4f}, time={eval_time:.2f}s\"\n    )\n    return metrics\n\n\nif __name__ == \"__main__\":\n    # Quick manual check: python evaluator.py <program.py>\n    path = sys.argv[1] if len(sys.argv) > 1 else \"initial_program.py\"\n    import json\n\n    print(json.dumps(evaluate(path), indent=2))\n"
}
for name, content in embedded_files.items():
    (problem_dir / name).write_text(content)

config_template = "max_iterations: {iterations}\ncheckpoint_interval: 1\nlog_level: \"INFO\"\n\nllm:\n  primary_model: \"{model_alias}\"\n  primary_model_weight: 1.0\n  api_base: \"{api_base}\"\n  api_key: \"not-needed\"\n  temperature: {temperature}\n  top_p: 0.95\n  max_tokens: {max_tokens}\n  timeout: 600\n\nprompt:\n  system_message: |\n    You are an expert Python programmer and computational geometry assistant.\n    Improve run_packing() for packing {packing_n} circles in a unit square.\n    Preserve the run_packing() interface exactly: it must return centers, radii,\n    sum_radii. Prefer simple deterministic numpy code. Valid geometry matters\n    more than claiming a large objective value.\n  num_top_programs: 3\n  use_template_stochasticity: true\n\ndatabase:\n  population_size: 12\n  archive_size: 8\n  num_islands: 1\n  elite_selection_ratio: 0.3\n  exploitation_ratio: 0.7\n\nevaluator:\n  timeout: 60\n  cascade_evaluation: false\n  parallel_evaluations: 1\n  use_llm_feedback: false\n\ndiff_based_evolution: false\nallow_full_rewrites: true\n"
config_text = config_template.format(
    iterations=ITERATIONS,
    model_alias=MODEL_ALIAS,
    api_base=LLM_API_BASE,
    temperature=TEMPERATURE,
    max_tokens=MAX_TOKENS,
    packing_n=PACKING_N,
)
(problem_dir / "config_tutorial.yaml").write_text(config_text)

print("Wrote files:")
for path in sorted(problem_dir.iterdir()):
    print("-", path)

## 7. Verify Score Functions Without the LLM

The hacked program returns overlapping circles and reports a fake score of 999.

In [ ]:
import importlib.util
import os
from pathlib import Path

import pandas as pd

problem_dir = Path(WORKDIR) / "circle_packing"
evaluator_path = problem_dir / "evaluator.py"
initial_path = problem_dir / "initial_program.py"
hacked_path = problem_dir / "hacked_program.py"

spec = importlib.util.spec_from_file_location("tutorial_evaluator", evaluator_path)
ev = importlib.util.module_from_spec(spec)
spec.loader.exec_module(ev)

os.environ["PACKING_N"] = str(PACKING_N)
rows = []
for mode in ["hard_valid_sum", "soft_penalty", "intentionally_bad_reported_sum"]:
    os.environ["SCORE_MODE"] = mode
    for label, path in [("valid seed", initial_path), ("hacked invalid", hacked_path)]:
        metrics = ev.evaluate(str(path))
        rows.append({
            "score_mode": mode,
            "program": label,
            "combined_score": metrics.get("combined_score"),
            "sum_radii": metrics.get("sum_radii"),
            "reported_sum": metrics.get("reported_sum"),
            "validity": metrics.get("validity"),
            "overlap_penalty": metrics.get("overlap_penalty"),
            "boundary_penalty": metrics.get("boundary_penalty"),
        })

pd.DataFrame(rows)

## 8. Visualization Helpers

In [ ]:
import json
import os
import re
import subprocess
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import clear_output, display


def checkpoint_number(path):
    match = re.fullmatch(r"checkpoint_(\d+)", path.name)
    return int(match.group(1)) if match else -1


def stable_checkpoints(run_dir, min_age_seconds=1.0):
    checkpoints_dir = Path(run_dir) / "checkpoints"
    if not checkpoints_dir.exists():
        return []
    now = time.time()
    out = []
    for path in checkpoints_dir.glob("checkpoint_*"):
        program_path = path / "best_program.py"
        info_path = path / "best_program_info.json"
        if not program_path.exists() or not info_path.exists():
            continue
        if max(program_path.stat().st_mtime, info_path.stat().st_mtime) > now - min_age_seconds:
            continue
        try:
            info = json.loads(info_path.read_text())
        except Exception:
            continue
        out.append((checkpoint_number(path), path, info))
    return sorted(out, key=lambda x: x[0])


def checkpoint_history(run_dir):
    rows = []
    for number, path, info in stable_checkpoints(run_dir, min_age_seconds=0.0):
        metrics = info.get("metrics", {})
        rows.append({
            "checkpoint": number,
            "current_iteration": info.get("current_iteration", number),
            "program_birth_iteration": info.get("iteration"),
            "combined_score": metrics.get("combined_score"),
            "sum_radii": metrics.get("sum_radii"),
            "validity": metrics.get("validity"),
        })
    return pd.DataFrame(rows)


def execute_program(program_path, env=None, timeout=20):
    script = f"""
import importlib.util, json, numpy as np
spec = importlib.util.spec_from_file_location('program', {str(program_path)!r})
program = importlib.util.module_from_spec(spec)
spec.loader.exec_module(program)
centers, radii, sum_radii = program.run_packing()
print(json.dumps({{
    'centers': np.asarray(centers, dtype=float).tolist(),
    'radii': np.asarray(radii, dtype=float).tolist(),
    'sum_radii': float(sum_radii),
}}))
"""
    result = subprocess.run([sys.executable, "-c", script], capture_output=True, text=True, timeout=timeout, env=env)
    if result.returncode != 0:
        raise RuntimeError(result.stderr[-2000:])
    return json.loads(result.stdout)


def plot_packing(centers, radii, title):
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.set_aspect("equal")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.add_patch(plt.Rectangle((0, 0), 1, 1, fill=False, linewidth=2))
    for (x, y), r in zip(centers, radii):
        ax.add_patch(plt.Circle((x, y), r, fill=False, linewidth=1.5))
        ax.plot([x], [y], marker=".", markersize=3)
    ax.set_title(title)
    plt.show()


def show_run_state(run_dir, env=None):
    hist = checkpoint_history(run_dir)
    latest = stable_checkpoints(run_dir)
    if hist.empty:
        print("No stable checkpoint yet.")
        return
    display(hist.tail(10))
    fig, ax = plt.subplots(figsize=(6, 3))
    ax.plot(hist["current_iteration"], hist["combined_score"], marker="o", label="combined_score")
    ax.plot(hist["current_iteration"], hist["sum_radii"], marker="x", label="sum_radii")
    ax.set_xlabel("current_iteration")
    ax.set_ylabel("score")
    ax.grid(True, alpha=0.3)
    ax.legend()
    plt.show()
    number, checkpoint_dir, info = latest[-1]
    print("Latest stable checkpoint:", checkpoint_dir.name)
    print("Metrics:", info.get("metrics", {}))
    try:
        data = execute_program(checkpoint_dir / "best_program.py", env=env)
        plot_packing(np.asarray(data["centers"]), np.asarray(data["radii"]), f"Best at checkpoint {number}")
    except Exception as exc:
        print("Could not execute best_program.py:", exc)

## 9. Run OpenEvolve and Watch Checkpoints

In [ ]:
import datetime as dt
import os
import queue
import subprocess
import sys
import threading
import time
from pathlib import Path

problem_dir = Path(WORKDIR) / "circle_packing"
run_dir = Path(RUN_ROOT) / f"circle_n{PACKING_N}_{SCORE_MODE}_{dt.datetime.now().strftime('%Y%m%d_%H%M%S')}"
run_dir.parent.mkdir(parents=True, exist_ok=True)

env = os.environ.copy()
env.update({
    "PACKING_N": str(PACKING_N),
    "SCORE_MODE": SCORE_MODE,
    "OPENAI_API_KEY": "not-needed",
    "PYTHONPATH": f"/content/openevolve:{env.get('PYTHONPATH', '')}",
})

cmd = [
    sys.executable,
    "/content/openevolve/openevolve-run.py",
    str(problem_dir / "initial_program.py"),
    str(problem_dir / "evaluator.py"),
    "--config", str(problem_dir / "config_tutorial.yaml"),
    "--iterations", str(ITERATIONS),
    "--output", str(run_dir),
    "--api-base", LLM_API_BASE,
    "--primary-model", MODEL_ALIAS,
    "--log-level", "INFO",
]
print("Run directory:", run_dir)
print("Command:", " ".join(cmd))

line_queue = queue.Queue()

def reader_thread(pipe):
    for line in iter(pipe.readline, ""):
        line_queue.put(line.rstrip())
    pipe.close()

proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
threading.Thread(target=reader_thread, args=(proc.stdout,), daemon=True).start()

recent_lines = []
last_refresh = 0
while proc.poll() is None:
    while not line_queue.empty():
        recent_lines.append(line_queue.get())
        recent_lines = recent_lines[-20:]
    if time.time() - last_refresh >= 5:
        clear_output(wait=True)
        print("OpenEvolve running...")
        print("Run directory:", run_dir)
        print("Recent output:")
        print("\n".join(recent_lines[-12:]) or "(no stdout yet; check logs after the run)")
        print()
        show_run_state(run_dir, env=env)
        last_refresh = time.time()
    time.sleep(1)

while not line_queue.empty():
    recent_lines.append(line_queue.get())

clear_output(wait=True)
print("OpenEvolve finished with exit code:", proc.returncode)
print("Run directory:", run_dir)
print("Recent output:")
print("\n".join(recent_lines[-20:]) or "(no stdout captured)")
print()
show_run_state(run_dir, env=env)

if proc.returncode != 0:
    log_files = sorted((run_dir / "logs").glob("*.log"))
    if log_files:
        print("\nLog tail:")
        print(log_files[-1].read_text(errors="replace")[-4000:])
    raise RuntimeError(f"OpenEvolve failed with exit code {proc.returncode}")

## 10. Re-render a Run Directory

In [ ]:
#@title Re-render a previous run
RUN_DIR_TO_RENDER = ""  #@param {type:"string"}
if RUN_DIR_TO_RENDER.strip():
    env = os.environ.copy()
    env.update({"PACKING_N": str(PACKING_N), "SCORE_MODE": SCORE_MODE})
    show_run_state(Path(RUN_DIR_TO_RENDER.strip()), env=env)
else:
    print("Paste a run directory path, e.g. /content/openevolve_runs/circle_n16_hard_valid_sum_...")

## 11. Fallback Visualization

Run this even if the LLM did not start. It demonstrates why the score function matters.

In [ ]:
import os
from pathlib import Path

problem_dir = Path(WORKDIR) / "circle_packing"
for mode, program_name in [
    ("hard_valid_sum", "initial_program.py"),
    ("hard_valid_sum", "hacked_program.py"),
    ("intentionally_bad_reported_sum", "hacked_program.py"),
    ("soft_penalty", "hacked_program.py"),
]:
    os.environ["PACKING_N"] = str(PACKING_N)
    os.environ["SCORE_MODE"] = mode
    program_path = problem_dir / program_name
    metrics = ev.evaluate(str(program_path))
    data = execute_program(program_path, env=os.environ.copy())
    print(f"{program_name} / {mode}: {metrics}")
    plot_packing(np.asarray(data["centers"]), np.asarray(data["radii"]), f"{program_name} / {mode}")

## 12. Suggested Experiments

1. Change `SCORE_MODE` to `intentionally_bad_reported_sum` and watch reward hacking.
2. Change `SCORE_MODE` to `soft_penalty` and compare it with the validity-gated score.
3. Change `PACKING_N` from `16` to `26`; the problem gets harder and slower.
4. Increase `TEMPERATURE` for more exploration.
5. Increase `MAX_TOKENS` if code extraction fails, or reduce it if generations are too slow.

The key lesson: OpenEvolve searches over code, but it follows the incentives created by your score function.